In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM


df = pd.read_csv(
    r"C:\A-CMSI research\Data\HMM data\HMM_Input_features.csv"
)

# =====================================
# Features
# =====================================

features = [
    "SPY_Return",
    "SPY_V",
    "RollingVol21",
    "RollingSkew21",
    "Drawdown",
    "VolOfVol",
    "VIX",
    "C_t"
]

X = df[features]


scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


best_model = None
best_score = -np.inf

for seed in range(30):

    model = GaussianHMM(
        n_components=6,
        covariance_type="full",
        n_iter=200,
        random_state=seed
    )

    model.fit(X_scaled)

    score = model.score(X_scaled)

    if score > best_score:

        best_score = score
        best_model = model

print("="*60)
print("Best Log-Likelihood:", best_score)
print("="*60)


hidden_states = best_model.predict(X_scaled)

df["State"] = hidden_states


print("\nSTATE MEANS\n")

print(
    df.groupby("State")[[
        "SPY_Return",
        "SPY_V",
        "RollingVol21",
        "RollingSkew21",
        "Drawdown",
        "VolOfVol",
        "VIX",
        "C_t"
    ]].mean()
)


print("\nTRANSITION MATRIX\n")

print(best_model.transmat_)


logL = best_model.score(X_scaled)

n_states = best_model.n_components
n_features = X_scaled.shape[1]
n_samples = X_scaled.shape[0]


k = (
    (n_states - 1)
    + n_states * (n_states - 1)
    + n_states * n_features
    + n_states * n_features * (n_features + 1) / 2
)

AIC = -2 * logL + 2 * k

BIC = -2 * logL + np.log(n_samples) * k

print("\nMODEL SELECTION")

print("----------------------------")
print("Number of States :", n_states)
print("Log-Likelihood   :", logL)
print("Parameters       :", int(k))
print("AIC              :", AIC)
print("BIC              :", BIC)
print("----------------------------")

Best Log-Likelihood: -11740.12578856552

STATE MEANS

       SPY_Return     SPY_V  RollingVol21  RollingSkew21  Drawdown  VolOfVol  \
State                                                                          
0       -0.000684  1.062213      0.013555      -0.398700 -0.063134  0.001903   
1        0.001500 -0.924892      0.005516      -0.077493 -0.002397  0.001122   
2        0.001156 -0.587662      0.005715      -0.040155 -0.004213  0.000627   
3       -0.000178  0.109992      0.007303      -0.484964 -0.022090  0.001137   
4        0.001986  0.727435      0.024078       0.213577 -0.114972  0.007192   
5        0.000554 -0.357242      0.010971       0.065252 -0.116730  0.001019   

             VIX       C_t  
State                       
0      23.582044  0.809754  
1      15.968514  0.521727  
2      12.662544  0.541795  
3      16.759696  0.617764  
4      31.767080  0.806139  
5      19.888578  0.733298  

TRANSITION MATRIX

[[9.71735055e-001 8.01151376e-003 2.52305895e-015 6.2

In [2]:
print(df["State"].value_counts().sort_index())

State
0    504
1    471
2    511
3    461
4    137
5    429
Name: count, dtype: int64
